# SQL Practice Questions - Part 1 (Questions 1-15)

## Introduction
This notebook contains the first 15 "Medium" difficulty SQL questions from the "Solve 50 SQL Questions in 2 Hours" challenge. 
These questions are designed to test your understanding of intermediate SQL concepts.

## Database Schema

We use a Hospital database with the following tables:

- **`patients`**: Info about patients (ID, Name, Gender, DOB, City, Province, etc.)
- **`admissions`**: Info about hospital admissions (Patient ID, Date, Diagnosis, Doctor ID)
- **`doctors`**: Info about doctors (ID, Name, Speciality)
- **`province_names`**: lookup for province names (ID, Name)

### ER Diagram
```mermaid
erDiagram
    patients ||--o{ admissions : "has"
    patients }|--|| province_names : "lives in"
    admissions }|--|| doctors : "attended by"

    patients {
        int patient_id PK
        string first_name
        string last_name
        string gender
        date birth_date
        string city
        string province_id
        string allergies
        int height
        int weight
    }
    admissions {
        int patient_id FK
        date admission_date
        date discharge_date
        string diagnosis
        int attending_doctor_id FK
    }
    doctors {
        int doctor_id PK
        string first_name
        string last_name
        string speciality
    }
    province_names {
        string province_id PK
        string province_name
    }
```

## Local Data Setup
Run the following cell to create the database in-memory for practice.

In [4]:
import sqlite3
import pandas as pd

# Create in-memory database
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

# Create Tables
cursor.executescript('''
    CREATE TABLE province_names (
        province_id TEXT PRIMARY KEY,
        province_name TEXT
    );

    CREATE TABLE patients (
        patient_id INTEGER PRIMARY KEY,
        first_name TEXT,
        last_name TEXT,
        gender TEXT,
        birth_date DATE,
        city TEXT,
        province_id TEXT,
        allergies TEXT,
        height INTEGER,
        weight INTEGER,
        FOREIGN KEY(province_id) REFERENCES province_names(province_id)
    );

    CREATE TABLE doctors (
        doctor_id INTEGER PRIMARY KEY,
        first_name TEXT,
        last_name TEXT,
        speciality TEXT
    );

    CREATE TABLE admissions (
        patient_id INTEGER,
        admission_date DATE,
        discharge_date DATE,
        diagnosis TEXT,
        attending_doctor_id INTEGER,
        FOREIGN KEY(patient_id) REFERENCES patients(patient_id),
        FOREIGN KEY(attending_doctor_id) REFERENCES doctors(doctor_id)
    );
''')

# Insert Dummy Data
cursor.executescript('''
    INSERT INTO province_names VALUES ('ON', 'Ontario'), ('BC', 'British Columbia'), ('AB', 'Alberta'), ('QC', 'Quebec');

    INSERT INTO patients (patient_id, first_name, last_name, gender, birth_date, city, province_id, allergies, height, weight) VALUES 
        (1, 'Donald', 'Waterfield', 'M', '1963-02-12', 'Barrie', 'ON', NULL, 156, 65),
        (2, 'Mickey', 'Baum', 'M', '1981-05-28', 'Toronto', 'ON', 'Penicillin', 178, 77),
        (3, 'Jabba', 'Smith', 'M', '1974-12-11', 'Vancouver', 'BC', NULL, 190, 110),
        (4, 'Minnie', 'Mouse', 'F', '1995-01-01', 'Calgary', 'AB', NULL, 160, 55),
        (5, 'Abby', 'Addams', 'F', '1978-02-02', 'Halifax', 'NS', 'Morphine', 165, 60),
        (6, 'Brian', 'Griffin', 'M', '2000-01-01', 'Quahog', 'AB', NULL, 120, 30);

    INSERT INTO doctors (doctor_id, first_name, last_name, speciality) VALUES
        (1, 'Claude', 'Walls', 'Internist'),
        (5, 'Joshua', 'Green', 'Pediatrician'),
        (19, 'Lisa', 'Cuddy', 'Endocrinologist');

    INSERT INTO admissions (patient_id, admission_date, discharge_date, diagnosis, attending_doctor_id) VALUES
        (1, '2019-01-01', '2019-01-04', 'Flu', 1),
        (2, '2019-01-02', '2019-01-05', 'Dementia', 5),
        (3, '2018-01-01', '2018-01-15', 'Pregnancy', 19),
        (3, '2019-02-02', '2019-02-05', 'Pregnancy', 19),
        (542, '2018-06-11', '2018-06-12', 'Pain', 1),
        (542, '2019-04-06', '2019-04-07', 'Abdominal Pain', 1);
''')

def run_query(query):
    return pd.read_sql_query(query, conn)
    
print("Database Setup Complete!")


Database Setup Complete!


## Questions

### Q1: Show unique birth years from patients and order them by ascending.

**Description**: Extract the year from the birth_date column and return only unique years, sorted from earliest to latest.

**Tags**: `DATE FUNCTION`, `DISTINCT`, `ORDER BY`

**Input/Output Example**:
Input (`birth_date`): `1980-01-01`, `1980-05-05`, `1990-12-12`  
Output (`birth_year`): `1980`, `1990`

**Hint**: 
Use a date function to extract the year. `DISTINCT` handles duplicate years.

**Logical Understanding**:
1.  Select `birth_date` from `patients`.
2.  Apply a function to get the year (e.g., `YEAR()` in some SQL dialects, or `STRFTIME` in SQLite).
3.  Apply `DISTINCT` to remove duplicates.
4.  `ORDER BY` the year in ascending order.

**Common Pitfalls**:
- Forgetting `DISTINCT` will result in one row per patient instead of unique years.
- Date function syntax varies by database (MySQL `YEAR()`, SQLite `STRFTIME('%Y', ...)`).

**Solution** (SQLite Compatible):

```sql
SELECT 
    DISTINCT STRFTIME('%Y', birth_date) as birth_year
FROM patients
ORDER BY birth_year ASC;
```

**Spark SQL / SQL Server Solution**:
```sql
SELECT 
    DISTINCT YEAR(birth_date) as birth_year
FROM patients
ORDER BY birth_year ASC;
```

In [ ]:
run_query("""
SELECT 
    DISTINCT STRFTIME('%Y', birth_date) as birth_year
FROM patients
ORDER BY birth_year ASC;
""")

### Q2: Show unique first names from the patients table which only occurs once in the list.

**Description**: Find first names that are unique among all patients (i.e., no other patient has that name).

**Tags**: `GROUP BY`, `HAVING`, `AGGREGATION`

**Input/Output Example**:
Input: 'John', 'John', 'Alice'  
Output: 'Alice'

**Hint**: 
Count how many times each name appears. Filter for counts equal to 1.

**Logical Understanding**:
1.  `SELECT` first_name.
2.  `GROUP BY` first_name.
3.  `HAVING` count(*) = 1.

**Solution** (Standard SQL - Works in Spark/SQL Server):
```sql
SELECT first_name
FROM patients
GROUP BY first_name
HAVING count(*) = 1;
```

In [3]:
run_query("""
SELECT first_name
FROM patients
GROUP BY first_name
HAVING count(*) = 1;
""")

,first_name
0,Abby
1,Brian
2,Donald
3,Jabba
4,Mickey
5,Minnie


### Q3: Show patient_id and first_name from patients where their first_name starts and ends with 's' and is at least 6 characters long.

**Description**: Filter string based on start/end characters and length.

**Tags**: `STRING FUNCTIONS`, `LIKE`, `LENGTH`, `FILTER`

**Input/Output Example**:
Input: 'Sarah', 'Seven', 'Silas'  
Output: 'Silas' (if length >= 6, but 'Silas' is 5... 'Stevens' would match)

**Hint**: 
Use `LIKE 's%s'` for the pattern. Use `LENGTH()` (or `LEN()`) for size check.

**Logical Understanding**:
1.  Filter where `first_name` like 's%s' (starts/ends with s).
2.  AND length check.

**Solution** (SQLite/Spark SQL):
```sql
SELECT patient_id, first_name
FROM patients
WHERE first_name LIKE 's%s'
  AND LENGTH(first_name) >= 6;
```

**SQL Server Solution**:
```sql
SELECT patient_id, first_name
FROM patients
WHERE first_name LIKE 's%s'
  AND LEN(first_name) >= 6;
```

In [2]:
run_query("""
SELECT patient_id, first_name
FROM patients
WHERE first_name LIKE 's%s'
  AND LENGTH(first_name) >= 6;
""")

,patient_id,first_name


### Q4: Show patient_id, first_name, last_name from patients whose diagnosis is 'Dementia'.

**Description**: Join patients and admissions to find patients with a specific diagnosis.

**Tags**: `JOIN`, `WHERE`

**Input/Output Example**:
Patients: [1, John], [2, Jane]  
Admissions: [1, Flu], [2, Dementia]  
Output: [2, Jane]

**Hint**: 
Join on `patient_id`. Filter `diagnosis`.

**Solution** (Standard SQL - Works in Spark/SQL Server):
```sql
SELECT 
    patients.patient_id, 
    first_name, 
    last_name
FROM patients
JOIN admissions ON patients.patient_id = admissions.patient_id
WHERE diagnosis = 'Dementia';
```

In [ ]:
run_query("""
SELECT 
    patients.patient_id, 
    first_name, 
    last_name
FROM patients
JOIN admissions ON patients.patient_id = admissions.patient_id
WHERE diagnosis = 'Dementia';
""")

### Q5: Display every patient's first_name. Order the list by the length of each name and then by alphabetically.

**Description**: Sort logic combining length and string value.

**Tags**: `ORDER BY`, `LENGTH`

**Input/Output Example**:
Input: 'Ann', 'Bo', 'Al'  
Output: 'Al', 'Bo', 'Ann' (Length 2 first, then alphabetical; Length 3 next)

**Hint**: 
Order by Length first, then Name.

**Solution** (SQLite/Spark SQL):
```sql
SELECT first_name
FROM patients
ORDER BY 
    LENGTH(first_name),
    first_name;
```

**SQL Server Solution**:
```sql
SELECT first_name
FROM patients
ORDER BY 
    LEN(first_name),
    first_name;
```

In [ ]:
run_query("""
SELECT first_name
FROM patients
ORDER BY 
    LENGTH(first_name),
    first_name;
""")

### Q6: Show the total amount of male patients and the total amount of female patients in the patients table. Display the two results in the same row.

**Description**: Pivot or Conditional Aggregation to show counts in columns instead of rows.

**Tags**: `CASE WHEN`, `SUM/COUNT`

**Input/Output Example**:
Output: `male_count: 50`, `female_count: 45`

**Hint**: 
Use `SUM(CASE WHEN gender='M' THEN 1 ELSE 0 END)`.

**Solution** (Standard SQL - Works in Spark/SQL Server):
```sql
SELECT 
    SUM(CASE WHEN gender = 'M' THEN 1 ELSE 0 END) as male_count,
    SUM(CASE WHEN gender = 'F' THEN 1 ELSE 0 END) as female_count
FROM patients;
```

**Alternative (Postgres/Spark SQL)**:
```sql
SELECT 
    count(1) FILTER (WHERE gender = 'M') as male_count,
    count(1) FILTER (WHERE gender = 'F') as female_count
FROM patients;
```

In [ ]:
run_query("""
SELECT 
    SUM(CASE WHEN gender = 'M' THEN 1 ELSE 0 END) as male_count,
    SUM(CASE WHEN gender = 'F' THEN 1 ELSE 0 END) as female_count
FROM patients;
""")

### Q7: Show first_name, last_name, and allergies from patients which have allergies to either 'Penicillin' or 'Morphine'. Show results ordered ascending by allergies then by first_name then by last_name.

**Description**: Filtering with `IN` or `OR` and multi-level sorting.

**Tags**: `WHERE`, `IN`, `ORDER BY`

**Hint**: 
Use `allergies IN (...)`.

**Solution** (Standard SQL - Works in Spark/SQL Server):
```sql
SELECT first_name, last_name, allergies
FROM patients
WHERE allergies IN ('Penicillin', 'Morphine')
ORDER BY allergies, first_name, last_name;
```

In [ ]:
run_query("""
SELECT first_name, last_name, allergies
FROM patients
WHERE allergies IN ('Penicillin', 'Morphine')
ORDER BY allergies, first_name, last_name;
""")

### Q8: Show patient_id, diagnosis from admissions. Find patients admitted multiple times for the same diagnosis.

**Description**: Identify duplicates based on a composite key (patient + diagnosis).

**Tags**: `GROUP BY`, `HAVING`

**Hint**: 
Group by Patient AND Diagnosis. Check if count > 1.

**Solution** (Standard SQL - Works in Spark/SQL Server):
```sql
SELECT patient_id, diagnosis
FROM admissions
GROUP BY patient_id, diagnosis
HAVING count(*) > 1;
```

In [ ]:
run_query("""
SELECT patient_id, diagnosis
FROM admissions
GROUP BY patient_id, diagnosis
HAVING count(*) > 1;
""")

### Q9: Show the city and the total number of patients in the city. Order from most to least patients and then by city name ascending.

**Description**: Basic Aggregation and Sorting.

**Tags**: `GROUP BY`, `COUNT`, `ORDER BY`

**Hint**: 
Group by city. Count rows. Sort by Count DESC, City ASC.

**Solution** (Standard SQL - Works in Spark/SQL Server):
```sql
SELECT city, count(*) as num_patients
FROM patients
GROUP BY city
ORDER BY num_patients DESC, city ASC;
```

In [ ]:
run_query("""
SELECT city, count(*) as num_patients
FROM patients
GROUP BY city
ORDER BY num_patients DESC, city ASC;
""")

### Q10: Show first_name, last_name and role of every person that is either patient or doctor. The roles are either "Patient" or "Doctor".

**Description**: Combine two tables into one list with a hardcoded discriminator column.

**Tags**: `UNION ALL`

**Hint**: 
Use `UNION ALL`.

**Solution** (Standard SQL - Works in Spark/SQL Server):
```sql
SELECT first_name, last_name, 'Patient' as role FROM patients
UNION ALL
SELECT first_name, last_name, 'Doctor' as role FROM doctors;
```

In [ ]:
run_query("""
SELECT first_name, last_name, 'Patient' as role FROM patients
UNION ALL
SELECT first_name, last_name, 'Doctor' as role FROM doctors;
""")

### Q11: Show all allergies ordered by popularity. Remove NULL values from query.

**Description**: Rank categorical data by frequency.

**Tags**: `GROUP BY`, `ORDER BY`, `NOT NULL`

**Hint**: 
Filter `IS NOT NULL`. Group by allergy.

**Solution** (Standard SQL - Works in Spark/SQL Server):
```sql
SELECT allergies, count(*) as popularity
FROM patients
WHERE allergies IS NOT NULL
GROUP BY allergies
ORDER BY popularity DESC;
```

In [ ]:
run_query("""
SELECT allergies, count(*) as popularity
FROM patients
WHERE allergies IS NOT NULL
GROUP BY allergies
ORDER BY popularity DESC;
""")

### Q12: Show all patients first_name, last_name, and birth_date who were born in the 1970s decade. Sort the list starting from the earliest birth_date.

**Description**: Date range filtering.

**Tags**: `DATE FILTERING`, `BETWEEN`

**Hint**: 
1970s means >= 1970-01-01 AND < 1980-01-01.

**Solution** (SQLite Analysis):
```sql
SELECT first_name, last_name, birth_date
FROM patients
WHERE STRFTIME('%Y', birth_date) >= '1970'
  AND STRFTIME('%Y', birth_date) < '1980'
ORDER BY birth_date ASC;
```

**Spark SQL / SQL Server Solution**:
```sql
SELECT first_name, last_name, birth_date
FROM patients
WHERE birth_date >= '1970-01-01' AND birth_date < '1980-01-01'
ORDER BY birth_date ASC;
-- OR using YEAR()
SELECT first_name, last_name, birth_date
FROM patients
WHERE YEAR(birth_date) BETWEEN 1970 AND 1979
ORDER BY birth_date ASC;
```

In [ ]:
run_query("""
SELECT first_name, last_name, birth_date
FROM patients
WHERE STRFTIME('%Y', birth_date) >= '1970'
  AND STRFTIME('%Y', birth_date) < '1980'
ORDER BY birth_date ASC;
""")

### Q13: We want to display each patient's full name in a single column. Their last_name in all upper letters must appear first, then first_name in all lower case letters. Separate the last_name and first_name with a comma. Order the list by the first_name in descending order.

**Description**: String manipulation (Upper, Lower, Concat).

**Tags**: `STRING FUNCTIONS`, `CONCAT`, `UPPER`, `LOWER`

**Hint**: 
concat upper last, comma, lower first.

**Solution** (SQLite):
```sql
SELECT UPPER(last_name) || ',' || LOWER(first_name) as full_name_format
FROM patients
ORDER BY first_name DESC;
```

**Spark SQL / SQL Server Solution**:
```sql
SELECT CONCAT(UPPER(last_name), ',', LOWER(first_name)) as full_name_format
FROM patients
ORDER BY first_name DESC;
-- SQL Server also supports '+' for strings:
-- UPPER(last_name) + ',' + LOWER(first_name)
```

In [ ]:
run_query("""
SELECT UPPER(last_name) || ',' || LOWER(first_name) as full_name_format
FROM patients
ORDER BY first_name DESC;
""")

### Q14: Show the province_id(s), sum of height; where the total sum of its patient's height is greater than or equal to 7,000.

**Description**: Aggregation with filtering on the result of the aggregation.

**Tags**: `GROUP BY`, `HAVING`, `SUM`

**Hint**: 
Use `HAVING`.

**Solution** (Standard SQL - Works in Spark/SQL Server):
```sql
SELECT province_id, SUM(height) as total_height
FROM patients
GROUP BY province_id
HAVING SUM(height) >= 7000;
```

In [ ]:
run_query("""
SELECT province_id, SUM(height) as total_height
FROM patients
GROUP BY province_id
HAVING total_height >= 7000;
""")

### Q15: Show the difference between the largest weight and smallest weight for patients with the last name 'Maroni'.

**Description**: Difference between aggregates.

**Tags**: `MAX`, `MIN`, `MATH`

**Hint**: 
`MAX - MIN`.

**Solution** (Standard SQL - Works in Spark/SQL Server):
```sql
SELECT MAX(weight) - MIN(weight) as weight_diff
FROM patients
WHERE last_name = 'Maroni'; 
```

In [ ]:
run_query("""
SELECT MAX(weight) - MIN(weight) as weight_diff
FROM patients
WHERE last_name = 'Maroni'; 
""")